## Visualização

In [ ]:
def plot_accuracy(corrects, errors):
    episodes = np.arange(len(corrects))
    plt.figure(figsize=(12, 6))
    plt.plot(episodes, corrects, label='Acertos')
    plt.plot(episodes, errors, label='Erros')
    plt.xlabel('Episódios')
    plt.ylabel('Número de Acertos/Erros')
    plt.title('Evolução de Acertos e Erros ao Longo dos Episódios')
    plt.legend()
    plt.show()

In [ ]:
def plot_policy(policy, shape):
    grid = np.zeros(shape)
    for state in range(len(policy)):
        grid[state // shape[1], state % shape[1]] = policy[state]

    plt.figure(figsize=(8, 8))
    plt.imshow(grid, cmap='viridis', interpolation='nearest')
    plt.colorbar()
    plt.title('Política Final (Ação Preferida por Estado)')
    plt.show()

In [ ]:
def plot_rewards(rewards):
    plt.figure(figsize=(12, 6))
    plt.plot(rewards)
    plt.xlabel("Episódios")
    plt.ylabel("Recompensa")
    plt.title(f"Recompensa por episódio")
    plt.grid(True)
    plt.show()

In [ ]:
def plot_state_visitations(data, plot_titles, idx):
    data_keys = ["state_visits_before", "state_visits_after"]
    positions = [211, 212]
    titles = plot_titles
    wall_ends = [None, -1]

    for i in range(2):
        state_visits = data[data_keys[i]][idx]
        average_state_visits = np.mean(state_visits, axis=0)
        num_states = len(average_state_visits)  # Get the number of states dynamically

        if num_states == 16:  # For Frozen Lake 4x4
            grid_shape = (4, 4)
        else:
            raise ValueError("Unsupported grid shape")

        # Reshape to match Frozen Lake grid shape (4x4)
        grid_state_visits = np.rot90(average_state_visits.reshape(grid_shape).T)

        plt.subplot(positions[i])
        plt.pcolormesh(grid_state_visits, edgecolors='gray', linewidth=1, cmap='viridis')
        plt.text(0 + 0.5, 0 + 0.5, 'S', horizontalalignment='center', verticalalignment='center', color='red')
        plt.text(3 + 0.5, 3 + 0.5, 'G', horizontalalignment='center', verticalalignment='center', color='green')
        plt.title(titles[i])
        plt.axis('off')
        cm = plt.get_cmap()
        cm.set_bad('gray')

    plt.subplots_adjust(bottom=0.0, right=0.7, top=1.0)
    cax = plt.axes([1., 0.0, 0.075, 1.])
    cbar = plt.colorbar(cax=cax)
    plt.show()

In [ ]:
# Plotar resultados
def plot_results(all_reward_sums, step_sizes, algorithms):
    for algorithm in algorithms:
        algorithm_means = np.array([np.mean(all_reward_sums[(algorithm, step_size)]) for step_size in step_sizes])
        algorithm_stds = np.array([np.std(all_reward_sums[(algorithm, step_size)]) for step_size in step_sizes])
        plt.plot(step_sizes, algorithm_means, marker='o', linestyle='solid', label=algorithm)
        plt.fill_between(step_sizes, algorithm_means + algorithm_stds, algorithm_means - algorithm_stds, alpha=0.2)

    plt.legend()
    plt.xlabel("Step-size")
    plt.ylabel("Sum of\n rewards\n per episode", rotation=0, labelpad=50)
    plt.xticks(step_sizes)
    plt.show()

In [ ]:
def plot_rewards_cumulative(rewards):
    cumulative_rewards = [sum(rewards[:i+1]) for i in range(len(rewards))]

    plt.figure(figsize=(12, 6))
    plt.plot(cumulative_rewards)
    plt.xlabel("Episódios")
    plt.ylabel("Recompensa Acumulada")
    plt.title("Recompensa Acumulada ao Longo dos Episódios")
    plt.grid(True)
    plt.show()

## Geraradores

In [ ]:
def test_agent(env, agent, goal, epsilon_final=0.01, epsilon_decay=0.01):
    """
    Testa o agente no ambiente até que o total de recompensas atinja o objetivo.

    Parâmetros:
    env: O ambiente do Gym.
    agent: O agente que está sendo testado.
    goal: O objetivo de recompensas totais que deve ser alcançado.
    epsilon_final: O valor final de epsilon para exploração.
    epsilon_decay: A taxa de decaimento de epsilon por episódio.

    Retorna:
    list_rewards: Uma lista com as recompensas acumuladas por episódio.
    """

    total_rewards = 0  # Inicializa o total de recompensas
    list_rewards = []  # Inicializa a lista de recompensas acumuladas
    episodes = 0  # Contador de episódios

    # Jogar episódios até que as recompensas totais alcancem o objetivo
    while total_rewards < goal:
        obs = env.reset()  # Reiniciar o ambiente para um novo episódio
        action = agent.agent_start(obs)  # Iniciar o agente com a observação inicial
        done = False  # Reiniciar a flag de término para cada episódio
        episode_reward = 0  # Reiniciar a recompensa do episódio

        # Loop através de cada passo no episódio
        while not done:
            obs, reward, terminated, _ = env.step(action)  # Executar um passo no ambiente
            episode_reward += reward  # Acumular recompensa do episódio

            if terminated:
                agent.agent_end(reward)  # Encerrar o episódio do agente
                done = True  # Definir a flag de término para sair do loop
            else:
                action = agent.agent_step(reward, obs)  # Continuar o passo do agente

        total_rewards += episode_reward  # Acumular recompensas
        episodes += 1  # Contar os episódios
        list_rewards.append(total_rewards)  # Adicionar recompensas totais à lista

        # Decaimento do epsilon após cada episódio
        if agent.epsilon > epsilon_final:
            agent.epsilon = max(agent.epsilon - epsilon_decay, epsilon_final)

    # Fechar o ambiente
    env.close()

    # Imprimir resultados finais
    print(f"Episódios: {episodes}")
    print(f"Precisão: {episodes/total_rewards :.2f}")

    return list_rewards

In [ ]:
def run_episodes(env, agent, total_episodes, max_steps):
    rewards = []

    for episode in tqdm(range(total_episodes), desc="Episódios de treinamento"):
        observation = env.reset()
        done = False
        total_reward = 0
        steps = 0

        while not done and steps < max_steps:
            if steps == 0:
                action = agent.agent_start(observation)
            else:
                action = agent.agent_step(reward, observation)

            observation, reward, done, info = env.step(action)
            total_reward += reward
            steps += 1

        agent.agent_end(reward)
        rewards.append(total_reward)

    return rewards

In [ ]:
def run_episode(env, agent, max_steps):
    state = env.reset()
    total_reward = 0
    for _ in range(max_steps):
        action = agent.agent_start(state) if _ == 0 else agent.agent_step(reward, state)
        state, reward, done, _ = env.step(action)
        total_reward += reward
        if done:
            agent.agent_end(reward)
            break
    return total_reward

In [ ]:
# Função para executar o experimento
def run_experiment(env_name, agents, num_episodes, step_sizes, num_runs, max_steps):
    env = gym.make(env_name)
    all_reward_sums = {}
    cross_product = list(product(agents.keys(), step_sizes, range(num_runs)))

    for agent_name, step_size, run in tqdm(cross_product, desc="Executando experimentos"):
        if (agent_name, step_size) not in all_reward_sums:
            all_reward_sums[(agent_name, step_size)] = []

        agent_class = agents[agent_name]
        agent_info = {
            "env": env,
            "num_actions": env.action_space.n,
            "num_states": env.observation_space.n,
            "epsilon": 0.1,
            "discount": 1.0,
            "step_size": step_size,
            "seed": run
        }
        agent = agent_class(**agent_info)

        total_rewards = []
        for episode in range(num_episodes):
            total_reward = run_episode(env, agent, max_steps)
            total_rewards.append(total_reward)
        all_reward_sums[(agent_name, step_size)].append(np.mean(total_rewards))

    env.close()
    return all_reward_sums, step_sizes